In [ ]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
# 1. Configuration
TRAIN_PATH = "project_1_missing_metadata_older_by_event_date.csv"
VAL_PATH = "project_1_missing_metadata_newer_by_event_date.csv"

DATE_COLUMN = "EventDate"
TOP_N_IDS = 50
MAX_TRAIN_ROWS_PER_ID = 2_000
MAX_VALIDATION_ROWS_PER_ID = 500
RANDOM_SEED = 42

# 2. Load Datasets
df_train = pd.read_csv(TRAIN_PATH, low_memory=False)
df_val = pd.read_csv(VAL_PATH, low_memory=False)

df_train.columns = df_train.columns.str.strip()
df_val.columns = df_val.columns.str.strip()

# Standardize AnimalId column name
def detect_target_col(df):
    for col in ["AnimalId", "AnimalID", "animal_id", "AnimalNumber"]:
        if col in df.columns:
            return col
    return None

TARGET = "AnimalId"
df_train.rename(columns={detect_target_col(df_train): TARGET}, inplace=True)
df_val.rename(columns={detect_target_col(df_val): TARGET}, inplace=True)

# Format Target as clean string
def clean_id_column(series):
    numeric_series = pd.to_numeric(series, errors='coerce')
    cleaned = series.astype(str).str.strip()
    mask = numeric_series.notna() & (numeric_series % 1 == 0)
    cleaned[mask] = numeric_series[mask].astype(int).astype(str)
    return cleaned

df_train[TARGET] = clean_id_column(df_train[TARGET])
df_val[TARGET] = clean_id_column(df_val[TARGET])

# Remove invalid target rows
df_train = df_train[~df_train[TARGET].isin(["nan", "None", "", "<NA>"])].copy()
df_val = df_val[~df_val[TARGET].isin(["nan", "None", "", "<NA>"])].copy()

# Parse EventDate
if DATE_COLUMN in df_train.columns:
    df_train[DATE_COLUMN] = pd.to_datetime(df_train[DATE_COLUMN], errors="coerce")
if DATE_COLUMN in df_val.columns:
    df_val[DATE_COLUMN] = pd.to_datetime(df_val[DATE_COLUMN], errors="coerce")

# 3. Define Features
numerical_features = [
    "LactationNumber", "DaysInMilk", "Avgmilkflow",
    "Flow30_60Session", "YieldFirst2Min_Session",
    "YieldSession", "DurationSession_sec"
]
categorical_features = ["ReproductionStatus", "milking"]

num_cols = [c for c in numerical_features if c in df_train.columns]
cat_cols = [c for c in categorical_features if c in df_train.columns]

for col in cat_cols:
    df_train[col] = df_train[col].astype(str).str.strip()
    df_val[col] = df_val[col].astype(str).str.strip()

# Extract cyclical date features (NO event_year to prevent target leakage)
def add_date_features(df):
    df = df.copy()
    if DATE_COLUMN in df.columns and pd.api.types.is_datetime64_any_dtype(df[DATE_COLUMN]):
        df["event_month"] = df[DATE_COLUMN].dt.month
        df["event_day_of_year"] = df[DATE_COLUMN].dt.dayofyear
        df["event_day_of_week"] = df[DATE_COLUMN].dt.dayofweek
    return df

df_train = add_date_features(df_train)
df_val = add_date_features(df_val)

date_cols = [c for c in ["event_month", "event_day_of_year", "event_day_of_week"] if c in df_train.columns]
all_features = num_cols + cat_cols + date_cols

# 4. Filter Top N Overlapping IDs (Cows that exist in both past & future datasets)
common_ids = set(df_train[TARGET].unique()).intersection(set(df_val[TARGET].unique()))
train_counts = df_train[df_train[TARGET].isin(common_ids)][TARGET].value_counts()
selected_ids = train_counts.head(TOP_N_IDS).index.tolist()

df_train_filtered = df_train[df_train[TARGET].isin(selected_ids)].copy()
df_val_filtered = df_val[df_val[TARGET].isin(selected_ids)].copy()

# 5. Cap Rows per ID
def cap_rows_per_id(frame, row_cap):
    shuffled = frame.sample(frac=1, random_state=RANDOM_SEED)
    return shuffled.loc[shuffled.groupby(TARGET).cumcount() < row_cap].copy()

train_sample = cap_rows_per_id(df_train_filtered, MAX_TRAIN_ROWS_PER_ID)
val_sample = cap_rows_per_id(df_val_filtered, MAX_VALIDATION_ROWS_PER_ID)

# 6. CatBoost Training
train_pool = Pool(
    data=train_sample[all_features],
    label=train_sample[TARGET],
    cat_features=cat_cols if len(cat_cols) > 0 else None
)

val_pool = Pool(
    data=val_sample[all_features],
    label=val_sample[TARGET],
    cat_features=cat_cols if len(cat_cols) > 0 else None
)

model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.08,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=RANDOM_SEED,
    allow_writing_files=False,
    verbose=100
)

model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=40, use_best_model=True)

# 7. Model Accuracy Evaluation
val_preds = model.predict(val_pool).ravel()
true_ids = val_sample[TARGET]

acc = accuracy_score(true_ids, val_preds)
macro_f1 = f1_score(true_ids, val_preds, average="macro", zero_division=0)

print("\n" + "=" * 40)
print(f"Validation Accuracy: {acc * 100:.2f}%")
print(f"Validation Macro F1: {macro_f1:.4f}")
print("=" * 40)